In [ ]:
# Block 1: notebook description and analysis objective

#This notebook is being used to evaluate momentum, efficiency, relative performance, and factor exposure for a single asset.
#Original Risk Analysis blocks included here: 14-21.


In [ ]:
# Block 2: import libraries and initialize analytics services
import logging
import warnings
from pathlib import Path
import sys
import numpy as np
import pandas as pd
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.visualization import (
    Plotter,
    )
from Quantapp.visualization.core import (
    configure_plotly_notebook_renderers,
    )
from Quantapp.visualization.views.single_asset_profile.pricing.momentum_efficiency import (
    plot_benchmark_zscore_detail,
    plot_candlestick_drawdown_recovery_view,
    plot_momentum_zscore_comparison,
    plot_multi_benchmark_sharpe_spread_summary,
    plot_momentum_window_diagnostics_grid_view,
    plot_rolling_correlation_view,
    plot_seasonality_stack_view,
    plot_sharpe_sortino_comparison,
    plot_sharpe_surface_view,
    plot_sharpe_zscore_heatmap_view,
    plot_vix_fix_bands,
    )
from Quantapp.analytics import compute
from Quantapp.analytics import (
    Metric,
    MomentumAnalytics,
    RiskDistributionAnalytics,
    RiskRelativeAnalytics,
    SeriesTransforms,
    TimeSeriesAnalytics as Rolling,
    )
from Quantapp.data import get_market_history

warnings.filterwarnings("ignore")
logger = logging.getLogger("yfinance")
metric = Metric()
rolling = Rolling()
series_transforms = SeriesTransforms()
momentum_analytics = MomentumAnalytics()
risk_distribution_analytics = RiskDistributionAnalytics()
risk_relative_analytics = RiskRelativeAnalytics()

In [ ]:
# Block 3: initialize plotting helpers and Momentum & Efficiency display theme

qp = Plotter()

# This is intentionally notebook-local: importing Quantapp.visualization does not apply this theme.
configure_plotly_notebook_renderers()

In [ ]:
# Block 4: set notebook parameters

ticker_str = "SOXL"
vix_str = "^VIX"
interval = "1d"
period = "20y"
risk_free_ticker = "^IRX"
benchmark_tickers = ["SPY"]
time_frame_map = {"short": 21, "mid": 50, "long": 200}
selected_time_frame = [21, 50, 200]
length_of_plots = 20
var_position_value = None

In [ ]:
# Block 5: fetch market history and assign notebook roles
requested_symbols = [
    ticker_str,
    vix_str,
    risk_free_ticker,
    *benchmark_tickers,
]

asset_histories = get_market_history(
    symbols=requested_symbols,
    period=period,
    interval=interval,
    provider="yfinance",
    align=True,
)

asset_history           = asset_histories.get(ticker_str, pd.DataFrame())
vix_history             = asset_histories.get(vix_str, pd.DataFrame())
risk_free_proxy_history = asset_histories.get(risk_free_ticker, pd.DataFrame())

benchmark_data = {
    symbol: frame
    for symbol, frame in asset_histories.items()
    if symbol not in {ticker_str, vix_str, risk_free_ticker}
}

#loaded_benchmark_tickers = list(benchmark_data)
#analysis_index = asset_history.index


In [ ]:
# Block 7: derive analysis series from normalized market data

risk_free_daily_rate = series_transforms.annualized_yield_to_periodic_rate(
    risk_free_proxy_history,
    annualization_factor=252,
    input_is_percent=True,
    lag_periods=1,
    reference_index=asset_history.index,
)

ticker_monthly_data = series_transforms.resample(asset_history, frequency="monthly")
ticker_weekly_data = series_transforms.resample(asset_history, frequency="weekly")
ticker_daily_data = series_transforms.resample(asset_history, frequency="daily")

ticker_monthly_returns = ticker_monthly_data["Close"].pct_change(fill_method=None).dropna()
ticker_weekly_returns = ticker_weekly_data["Close"].pct_change(fill_method=None).dropna()
ticker_daily_returns = ticker_daily_data["Close"].pct_change(fill_method=None).dropna()


In [ ]:
# Block 9: build VIX Fix series and overlay standard deviation bands

#Volatility: VIX FIX

ticker_vix_fix = compute.rolling(asset_history['Close'], metric=metric.vix_fix, window=22)
benchmark_vix_fix = {
    symbol: compute.rolling(frame['Close'], metric=metric.vix_fix, window=22)
    for symbol, frame in benchmark_data.items()
}

fig = plot_vix_fix_bands(
    ticker_vix_fix,
    title='VIX Fix with Mean and Standard Deviations',
    stdev_values=[-0.5, 0.5, 1.5, 3],
)
fig.show()



In [ ]:
# Block 10: stack candlestick, drawdown comparison, and rolling recovery time
# Change selected_time_frame in Block 4 to a list like [21, 50, 200], then rerun this cell.

drawdown_recovery_context = risk_distribution_analytics.build_risk_distribution_context(
    close_series=asset_history['Close'],
    windows=selected_time_frame,
)

fig = plot_candlestick_drawdown_recovery_view(
    price_frame=asset_history,
    metrics_by_window=drawdown_recovery_context['metrics_by_window'],
    window_options=drawdown_recovery_context['windows'],
    default_window=drawdown_recovery_context['default_window'],
    show_window_menu=len(drawdown_recovery_context['windows']) > 1,
    ticker_label=ticker_str,
    candlestick_period=period,
    default_timeframe_label='10 Years',
)
fig.show()


In [ ]:
# Block 11: compute rolling Sharpe windows, momentum histograms, and volatility

window_sizes = list(range(3, 201))

momentum_diagnostics_context = momentum_analytics.build_momentum_window_diagnostics_context(
    close_series=asset_history['Close'],
    window_sizes=window_sizes,
    highlight_windows=(7, 21, 50, 200),
    surface_years=10,
    risk_free_rate=risk_free_daily_rate,
 )

fig_momentum_window_diagnostics_grid = plot_momentum_window_diagnostics_grid_view(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)
fig_sharpe_surface = plot_sharpe_surface_view(
    diagnostics_context=momentum_diagnostics_context,
    ticker_label=ticker_str,
)

fig_momentum_window_diagnostics_grid.show()
fig_sharpe_surface.show()


In [ ]:
# Block 12: plot stacked rolling Sharpe z-score heatmaps for 1-200 day windows plus cross-window summaries

heatmap_windows = list(range(1, 201))
heatmap_time_frame_map = {f"window_{window}": window for window in heatmap_windows}

sharpe_heatmap_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_history["Close"],
    time_frame_map=heatmap_time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
)

heatmap_term_config_map = sharpe_heatmap_context["term_config_map"]
sharpe_zscore_series_by_window = {
    config["time_frame"]: config["sharpe_zscore"]
    for config in heatmap_term_config_map.values()
}
sharpe_zscore_heatmap = pd.DataFrame(sharpe_zscore_series_by_window).sort_index()
heatmap_matrix = sharpe_zscore_heatmap.T.reindex(heatmap_windows)

benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
    asset_sharpe_map=sharpe_heatmap_context["asset_sharpe_map"],
    asset_component_map=sharpe_heatmap_context["asset_component_map"],
    benchmark_metrics=sharpe_heatmap_context["benchmark_metrics"],
    spread_plot_data=sharpe_heatmap_context["spread_plot_data"],
    time_frame_map=heatmap_time_frame_map,
)

benchmark_heatmap_matrices = {}
for symbol in benchmark_plot_payload["benchmark_order"]:
    symbol_series_by_window = {
        window: benchmark_plot_payload["summary_zscore_map"].get(term_key, {}).get(symbol, pd.Series(dtype=float))
        for term_key, window in heatmap_time_frame_map.items()
    }
    symbol_heatmap = pd.DataFrame(symbol_series_by_window).sort_index().T.reindex(heatmap_windows)
    if symbol_heatmap.columns.size > 0:
        benchmark_heatmap_matrices[symbol] = symbol_heatmap

fig = plot_sharpe_zscore_heatmap_view(
    heatmap_matrix=heatmap_matrix,
    benchmark_heatmap_matrices=benchmark_heatmap_matrices,
    heatmap_windows=heatmap_windows,
    benchmark_order=benchmark_plot_payload["benchmark_order"],
    default_benchmark=benchmark_plot_payload["default_benchmark"],
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
# Block 13: visualize monthly, weekly, and daily seasonality patterns

fig_ticker_seasonality_stack = plot_seasonality_stack_view(
    monthly_returns=ticker_monthly_returns,
    weekly_returns=ticker_weekly_returns,
    daily_returns=ticker_daily_returns,
    ticker_label=ticker_str,
    as_of=asset_history.index.max(),
)
fig_ticker_seasonality_stack.show()

In [ ]:
# Block 14: compute Sharpe/Sortino ratios and spreads

import importlib
import Quantapp.analytics.risk_relative_analytics as risk_relative_analytics_module

# Refresh the analytics class in case the notebook kernel still holds an older signature.
importlib.reload(risk_relative_analytics_module)
risk_relative_analytics = risk_relative_analytics_module.RiskRelativeAnalytics()

asset_close = asset_history['Close']

risk_context = risk_relative_analytics.build_sharpe_sortino_context(
    analytics=rolling,
    asset_close=asset_close,
    time_frame_map=time_frame_map,
    benchmark_data=benchmark_data,
    risk_free_rate=risk_free_daily_rate,
    selected_time_frame=selected_time_frame,
)

asset_sharpe_map = risk_context['asset_sharpe_map']
asset_component_map = risk_context['asset_component_map']
asset_sortino_map = risk_context['asset_sortino_map']
asset_sharpe_sortino_spread_map = risk_context['asset_sharpe_sortino_spread_map']

benchmark_metrics = risk_context['benchmark_metrics']
benchmark_order = risk_context['benchmark_order']
default_benchmark = risk_context['default_benchmark']
spread_plot_data = risk_context['spread_plot_data']
term_config_map = risk_context['term_config_map']
selected_term_config_map = risk_context['selected_term_config_map']


In [ ]:
# Block 15: plot rolling correlation of the asset versus benchmarks

asset_daily_returns = asset_history['Close'].pct_change(fill_method=None)
correlation_term_order = [term for term in time_frame_map if time_frame_map.get(term) is not None]
correlation_benchmark_order = benchmark_order if benchmark_order else list(benchmark_data.keys())
rolling_correlation_map = {}

for term in correlation_term_order:
    window = int(time_frame_map[term])
    term_series_map = {}

    for symbol in correlation_benchmark_order:
        benchmark_frame = benchmark_data.get(symbol)
        if benchmark_frame is None or 'Close' not in benchmark_frame:
            continue

        benchmark_daily_returns = benchmark_frame['Close'].pct_change(fill_method=None)
        aligned_returns = pd.concat(
            [
                asset_daily_returns.rename('asset'),
                benchmark_daily_returns.rename(symbol),
            ],
            axis=1,
        ).dropna()
        if aligned_returns.empty:
            continue

        rolling_correlation_series = aligned_returns['asset'].rolling(window).corr(aligned_returns[symbol]).dropna()
        if rolling_correlation_series.empty:
            continue

        term_series_map[symbol] = rolling_correlation_series

    if term_series_map:
        rolling_correlation_map[term] = term_series_map

rolling_correlation_fig = plot_rolling_correlation_view(
    rolling_correlation_map=rolling_correlation_map,
    time_frame_map=time_frame_map,
    term_order=correlation_term_order,
    benchmark_order=correlation_benchmark_order,
    ticker_label=ticker_str,
)
rolling_correlation_fig.show()


In [ ]:
# Block 16: plot Sharpe & Sortino efficiency for the selected timeframe set
# Change selected_time_frame in Block 4 to a list like [21, 50, 200], then rerun the notebook.

fig = plot_sharpe_sortino_comparison(
    term_config_map=selected_term_config_map,
    ticker_label=ticker_str,
)
fig.show()


In [ ]:
# Block 17: render interactive momentum z-score comparisons

window_pairs = {
    "21 vs 50": (21, 50),
    "50 vs 200": (50, 200),
    "200 vs 400": (200, 400),
}

zscore_data = momentum_analytics.momentum_zscore_map(
    asset_history['Close'],
    window_pairs=window_pairs,
)

fig = plot_momentum_zscore_comparison(
    zscore_data=zscore_data,
    ticker_label=ticker_str,
    default_label="200 vs 400",
    default_time_label="3 Years",
    sigma_levels=(0.5, 1.0, 1.5),
)
fig.update_layout(height=850)
fig.show()


In [ ]:
# Block 18: combine risk-adjusted return and benchmark plots
# Requires the current kernel session to have fresh outputs from Blocks 2, 7, and 14.
# Change selected_time_frame in Block 4 to a list like [21, 50, 200], then rerun the notebook.

if benchmark_order:
    benchmark_plot_payload = risk_relative_analytics.build_benchmark_plot_payload(
        asset_sharpe_map=asset_sharpe_map,
        asset_component_map=asset_component_map,
        benchmark_metrics=benchmark_metrics,
        spread_plot_data=spread_plot_data,
        time_frame_map=risk_context['time_frame_map'],
        selected_time_frame=selected_time_frame,
    )

    summary_fig = plot_multi_benchmark_sharpe_spread_summary(
        summary_zscore_map=benchmark_plot_payload['summary_zscore_map'],
        time_frame_map=benchmark_plot_payload['time_frame_map'],
        ticker_label=ticker_str,
    )
    summary_fig.show()

    detail_fig = plot_benchmark_zscore_detail(
        detail_zscore_map=benchmark_plot_payload['detail_zscore_map'],
        benchmark_order=benchmark_plot_payload['benchmark_order'],
        time_frame_map=benchmark_plot_payload['time_frame_map'],
        ticker_label=ticker_str,
        default_benchmark=benchmark_plot_payload['default_benchmark'],
    )
    detail_fig.show()
else:
    print("No benchmark data available for benchmark comparison plots.")
